# Corrected v5 training on the frozen evaluation partitions

This new launch wrapper uses the preserved executed engine and the published image manifest and splits. The recorded study reused 60 laboratory random runs and completed 180 new runs. Several Kaggle GPU sessions may be required. This notebook does not recalculate embeddings or solve fresh partitions.

Attach `arjuntejaswi/plant-village` and `warcoder/potato-leaf-disease-dataset`, enable Internet and a GPU. Do not attach the supplemental Sankalana Healthy images. Use a fresh Python session and the recorded training package versions. Future training is not guaranteed to be bitwise identical. For a quick verification without training, run `reanalysis.ipynb` instead.

In [ ]:
import sys, subprocess
from pathlib import Path
REPO = Path.cwd() if (Path.cwd()/"code/reanalyze.py").is_file() else Path.cwd()/"potato_disease_paper"
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "corrected-v5",
                           "https://github.com/Gonder301/potato_disease_paper.git", str(REPO)])
if not (REPO/"SHA256SUMS.json").is_file():
    raise RuntimeError("Expected the corrected v5 deposit. Use a fresh directory or download tag corrected-v5.")
print("Repository:", REPO, flush=True)


In [ ]:
SESSION_HOURS = 8.0
RESERVE_MINUTES = 15.0
IMAGE_ROOT = "/kaggle/input"
RESUME_STATE = ""  # Set to an extracted RESULTS_closure directory from the previous session.
REUSE_ORIGINAL_LAB_RANDOM = True  # Same 60 reused runs as the recorded revision.
STATE_DIR = Path("/kaggle/working/frozen_v5_state")
OUTPUT_DIR = Path("/kaggle/working/RESULTS_closure")


In [ ]:
import importlib.metadata, json
recorded = json.loads((REPO/"analysis/environment.json").read_text())["packages"]
for name in ["torch", "torchvision"]:
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = "missing"
    if installed != recorded[name]:
        raise RuntimeError(f"Recorded {name}={recorded[name]}, installed={installed}. Configure the recorded PyTorch/CUDA environment before training; see docs/REPRODUCIBILITY.md.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO/"requirements-training.txt")])
print("Training dependencies checked. The runtime also verifies package versions and GPU availability.", flush=True)


## Prepare or resume

The default preserves the same 60 laboratory random predictions used in the revision. To run all 240 classifiers afresh, turn reuse off and choose new state/output directories; those new predictions are a separate reconstruction. Existing output is resumed without silently changing the protocol. A previous exported state takes precedence when RESUME_STATE is provided.

In [ ]:
if RESUME_STATE:
    state = Path(RESUME_STATE)
else:
    state = STATE_DIR
    if not state.exists():
        command = [sys.executable, str(REPO/"code/prepare_training.py"), "--output", str(state)]
        if REUSE_ORIGINAL_LAB_RANDOM:
            command.append("--reuse-lab-random")
        subprocess.check_call(command)
if not (state/"project_state.json").is_file():
    raise RuntimeError("State directory is missing project_state.json")
print("Using state:", state, flush=True)


## Execute

The runtime reports progress at image checks, model checks, batches, epochs, validation, checkpoint saves and completed runs. A heartbeat only shows that the process responds. The default budget targets a controlled pause after 7 h 45 min. Download the exported ZIP after each controlled session and attach it to resume. A forced Kaggle cancellation can prevent export; an interrupted partial epoch repeats.

In [ ]:
subprocess.check_call([sys.executable, "-u", str(REPO/"code/run_training.py"),
                       "--state", str(state), "--images", IMAGE_ROOT, "--output", str(OUTPUT_DIR),
                       "--session-hours", str(SESSION_HOURS), "--reserve-minutes", str(RESERVE_MINUTES)])


In [ ]:
from IPython.display import FileLink, display
archive = OUTPUT_DIR.parent/"results_closure_v5.zip"
if archive.is_file():
    display(FileLink(str(archive)))
else:
    print("No export exists; inspect the runtime error above before continuing.")
